# ImparTail — huấn luyện đối chứng: mặt nạ theo công thức (5)

Bài báo, mục 4.2.1, công thức (5): `L = (1-m)*L`, kèm câu *"A mask ... voids
the contribution of loss"* — nhãn bị che phải đóng góp **0**.

Mã công bố (`stsg_base.py:305-318`, nhánh BCE) dựng vector đích toàn 0 rồi chỉ
set 1 cho nhãn **sống sót**. Với `nn.BCELoss()` reduction `'mean'` trên toàn bộ
`max_len` phần tử, nhãn dương bị che thành nhãn **âm** và vẫn sinh gradient
(≈ +0,095), đồng thời mẫu số vẫn là `max_len` nên mọi gradient còn lại bị co
(15/17 = 0,88×).

Notebook chạy **hai nhánh cùng cấu hình, cùng hạt giống**, chỉ khác cờ
`--true_masked_bce`, rồi so R@K / mR@K / hR@K.

Giả thuyết: phần lớn đánh đổi head–tail mà bài báo gọi là *"although small"*
(thực tế 6–17%, Bảng 6) là **hệ quả của cài đặt**, không phải bản chất phương pháp.


In [ ]:
# ── 0. Môi trường thật đang có gì ───────────────────────────────────
import torch, sys, subprocess, os
print("python      :", sys.version.split()[0])
print("torch       :", torch.__version__)
print("CUDA của torch:", torch.version.cuda)
print("GPU         :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ")
print("compute cap :", torch.cuda.get_device_capability(0) if torch.cuda.is_available() else "-")
print("arch hỗ trợ :", torch.cuda.get_arch_list() if torch.cuda.is_available() else "-")
r = subprocess.run(["nvcc","--version"], capture_output=True, text=True)
print("nvcc        :", (r.stdout.strip().split("\n")[-1] if r.returncode==0 else "không có nvcc"))

In [ ]:
# ── 0b. Hạ torch cho khớp GPU ───────────────────────────────────────
# Log lần chạy trước: Kaggle cấp Tesla P100 (sm_60) nhưng cài sẵn torch 2.10
# chỉ hỗ trợ từ sm_70. Cùng lỗi đã gặp với APKGC, khắc phục bằng torch 2.4.1.
#
# ImparTail vốn viết cho torch 1.10+cu111 (2021). Không có bản cu111 nào chạy
# được trên CUDA 12, nên 2.4.1+cu121 là điểm gần nhất vừa hợp P100 vừa còn tồn
# tại. CUDA extension của fasterRCNN có dựng nổi trên API torch 2.4 hay không
# thì chưa biết — đó chính là thứ lần chạy này kiểm.
import torch, subprocess, sys
cap = torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (0,0)
need = f"sm_{cap[0]}{cap[1]}" not in torch.cuda.get_arch_list() if torch.cuda.is_available() else False
print(f"GPU sm_{cap[0]}{cap[1]} | torch {torch.__version__} | cần hạ: {need}")

if need:
    !pip uninstall -y -q torchvision torchaudio 2>&1 | tail -1
    !pip install -q --force-reinstall torch==2.4.1 torchvision==0.19.1 --index-url https://download.pytorch.org/whl/cu121 2>&1 | tail -3
    r = subprocess.run([sys.executable, "-c",
        "import torch;a=torch.randn(8,8,device='cuda');"
        "print('torch',torch.__version__);print('arch',torch.cuda.get_arch_list());"
        "print('matmul OK',float((a@a).sum()))"], capture_output=True, text=True)
    print(r.stdout or r.stderr[-500:])
    print("\n>>> torch đã đổi. Các ô sau chạy qua subprocess nên nhận bản mới.")
else:
    print("không cần hạ.")

In [ ]:
# ── 1. Dataset: dò theo tệp mốc, không đoán đường dẫn ───────────────
import os, glob
ROOT = "/kaggle/input"
def find(marker, isdir=False):
    for dp, dn, fn in os.walk(ROOT):
        if (marker in dn) if isdir else (marker in fn):
            return dp
    return None

ANN = find("object_bbox_and_relationship.pkl")
print("annotation :", ANN)
if ANN: print("  ", sorted(os.listdir(ANN)))

# khung hình: thư mục cha chứa nhiều thư mục con dạng <id>.mp4
# Dò ĐỆ QUY. Bản trước chỉ quét 2 mức (ROOT/* và ROOT/*/*) nên trượt: Kaggle
# gắn dataset ở /kaggle/input/datasets/<user>/<slug>/... — sâu 3-4 mức. Cùng
# lần chạy đó annotation tìm thấy (nhờ os.walk) còn khung hình thì không.
FRAME_DIRS = []
for dp, dn, fn in os.walk(ROOT):
    kids = [k for k in dn if k.endswith(".mp4")]
    if len(kids) > 10:
        FRAME_DIRS.append((dp, len(kids)))
        dn[:] = []          # không đi sâu vào 1600 thư mục con
for p, n in FRAME_DIRS:
    print(f"khung hình : {p}  ({n} video)")
print(f"\nTỔNG: {sum(n for _, n in FRAME_DIRS)} thư mục video")
if not (ANN and FRAME_DIRS):
    print("\nCÂY /kaggle/input (3 mức đầu):")
    for dp, dn, fn in os.walk(ROOT):
        depth = dp.replace(ROOT, "").count("/")
        if depth > 3: dn[:] = []; continue
        print("  " * depth, os.path.basename(dp) or ROOT,
              f"[{len(dn)} thư mục, {len(fn)} tệp]")
    raise SystemExit("thiếu dataset")

In [ ]:
# ── 2. Gộp các part khung hình thành một cây thư mục bằng symlink ───
# /kaggle/input chỉ đọc và mỗi part là một mount riêng, nên ImparTail không
# thấy được toàn bộ. Symlink rẻ hơn copy 60GB rất nhiều.
import os
AG = "/kaggle/working/data/ag"
os.makedirs(f"{AG}/frames", exist_ok=True)
os.makedirs(f"{AG}/annotations", exist_ok=True)

n = 0
for src, _ in FRAME_DIRS:
    for v in os.listdir(src):
        if not v.endswith(".mp4"): continue
        dst = f"{AG}/frames/{v}"
        if not os.path.exists(dst):
            os.symlink(f"{src}/{v}", dst); n += 1
print(f"đã liên kết {n} thư mục video -> {AG}/frames")

for f in os.listdir(ANN):
    dst = f"{AG}/annotations/{f}"
    if not os.path.exists(dst): os.symlink(f"{ANN}/{f}", dst)
print("annotations:", sorted(os.listdir(f"{AG}/annotations")))

In [ ]:
# ── 3. Mã nguồn ImparTail ───────────────────────────────────────────
%cd /kaggle/working
!rm -rf ImparTail
!git clone -q https://github.com/rohithpeddi/ImparTail.git
%cd /kaggle/working/ImparTail
import os
print("có:", [f for f in os.listdir(".") if f.endswith((".py",".sh"))][:6])

In [ ]:
# ── 4. Cython: CẢ HAI extension ────────────────────────────────────
# Repo có hai .pyx phải build, không phải một. Bỏ sót bbox thì script chết ở
#   ModuleNotFoundError: No module named 'lib.fpn.box_intersections_cpu.bbox'
# nhưng mãi sau khi đã qua hết phần vá — nên rất tốn một vòng chạy.
# Danh sách này lấy bằng `find . -name "*.pyx"` ở máy, không phải theo lỗi.
import subprocess, sys, os, glob
ROOT = "/kaggle/working/ImparTail"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "cython"],
               capture_output=True, text=True)

# ── Bí danh numpy đã bị gỡ ──────────────────────────────────────────
# np.float / np.bool bị NumPy gỡ hẳn ở 1.24. Quét toàn repo: 13 chỗ / 8 tệp.
# PHẢI vá TRƯỚC khi cythonize — lỗi nằm trong bbox.pyx và chỉ lộ lúc import
# module đã biên dịch, nên vá sau thì vô ích.
#!/usr/bin/env python3
"""Thay các bí danh numpy đã bị gỡ (np.float, np.bool, …).

NumPy gỡ hẳn `np.float`, `np.int`, `np.bool`, `np.object`, `np.str` ở bản 1.24
(deprecated từ 1.20). Mã ImparTail viết trước đó nên còn dùng, và script đánh giá
dừng ở:

    AttributeError: module 'numpy' has no attribute 'float'

Đây chỉ là bí danh của kiểu dựng sẵn Python — thay bằng chính kiểu đó là tương
đương tuyệt đối, không đổi ngữ nghĩa hay độ chính xác.

Quét toàn repo cho ra 13 chỗ trong 8 tệp, gồm cả `.pyx` (phải vá TRƯỚC khi
cythonize, vì lỗi chỉ lộ ra lúc import module đã biên dịch).

    python fix_numpy_alias.py <đường dẫn ImparTail>
"""
import os, re, sys

#: np.float -> float. Không đụng np.float32/float64 nhờ ranh giới \b và (?!\d).
ALIASES = {"float": "float", "int": "int", "bool": "bool",
           "object": "object", "str": "str", "long": "int", "complex": "complex"}
PAT = re.compile(r"\bnp\.(" + "|".join(ALIASES) + r")\b(?!\d|_)")


def patch(path):
    try:
        src = open(path, encoding="utf-8").read()
    except (OSError, UnicodeDecodeError):
        return 0
    out = PAT.sub(lambda m: ALIASES[m.group(1)], src)
    if out == src:
        return 0
    open(path, "w", encoding="utf-8").write(out)
    n = len(PAT.findall(src))
    print(f"  ✓ {os.path.relpath(path):52} {n} chỗ")
    return n



_tot = 0
for _dp, _dn, _fn in os.walk(ROOT):
    if ".git" in _dp: continue
    for _f in _fn:
        if _f.endswith((".py", ".pyx")):
            _tot += patch(os.path.join(_dp, _f))
print(f"  vá {_tot} chỗ np.float/np.bool")


for sub in ("lib/draw_rectangles", "lib/fpn/box_intersections_cpu"):
    d = f"{ROOT}/{sub}"
    r = subprocess.run([sys.executable, "setup.py", "build_ext", "--inplace"],
                       cwd=d, capture_output=True, text=True)
    so = glob.glob(f"{d}/*.so")
    print(f"  {'✓' if so else '✗'} {sub}: {[os.path.basename(x) for x in so] or (r.stderr.strip().split(chr(10))[-1][:80])}")

In [ ]:
# ── 5. Vá THC rồi biên dịch CUDA extension ──────────────────────────
# Lỗi gốc lần trước:
#   ROIAlign_cuda.cu:5:10: fatal error: THC/THC.h: No such file or directory
# THC là header CUDA cũ của PyTorch, GỠ HẲN từ torch 1.11. Mã fasterRCNN viết
# thời torch 1.x nên còn include. Chỉ 5 ký hiệu THC được dùng, đều thay được:
#   THCCeilDiv -> impartail_ceil_div | THCudaCheck -> AT_CUDA_CHECK
#   THCudaMalloc/Free -> c10::cuda::CUDACachingAllocator::raw_alloc/raw_delete
import os, re, sys, subprocess

#!/usr/bin/env python3
"""Cập nhật CUDA extension của fasterRCNN cho torch hiện đại.

Vì sao cần: `THC/THC.h` là header CUDA cũ của PyTorch, **bị gỡ hẳn từ torch
1.11**. Mã fasterRCNN trong ImparTail viết thời torch 1.x nên còn include nó, và
biên dịch trên bất kỳ torch nào từ 1.11 trở lên đều dừng ở:

    ROIAlign_cuda.cu:5:10: fatal error: THC/THC.h: No such file or directory

Đây không phải lỗi khó — chỉ có năm ký hiệu THC được dùng, tất cả đều có thay
thế thẳng trong ATen:

    THC/THC.h, THCDeviceUtils.cuh  -> không cần, ATen đã có
    THC/THCAtomics.cuh            -> ATen/cuda/Atomic.cuh
    THCCeilDiv(a, b)              -> impartail_ceil_div (tự khai báo, xem HEADERS)
    THCudaCheck(x)                -> AT_CUDA_CHECK(x)
    THCudaMalloc(state, n)        -> c10::cuda::CUDACachingAllocator::raw_alloc(n)
    THCudaFree(state, p)          -> c10::cuda::CUDACachingAllocator::raw_delete(p)
    THCState *state = ...lazyInitCUDA()  -> bỏ, không còn dùng

    python fix_thc.py <đường dẫn ImparTail>
"""
import os, re, sys

#: Tự định nghĩa ceil_div thay vì dùng at::ceil_div. Lý do: ATen/ceil_div.h chỉ
#: có từ torch 2.5; trên 2.4.1 (bản duy nhất còn hỗ trợ sm_60 của Tesla P100)
#: thì at::ceil_div chưa tồn tại, và biên dịch dừng ở
#:   error: namespace "at" has no member "ceil_div"
#: Hàm chỉ có một dòng nên tự khai báo là chắc chắn nhất, không phụ thuộc bản torch.
#: Header xác minh trực tiếp trên nguồn tag v2.4.1 của pytorch, không đoán:
#:   AT_CUDA_CHECK        -> aten/src/ATen/cuda/Exceptions.h   (KHÔNG phải CUDAException.h)
#:   getCurrentCUDAStream -> c10/cuda/CUDAStream.h             (KHÔNG phải CUDAContext.h)
#:   raw_alloc/raw_delete -> c10/cuda/CUDACachingAllocator.h
#:   gpuAtomicAdd         -> aten/src/ATen/cuda/Atomic.cuh
HEADERS = """#include <ATen/cuda/CUDAContext.h>
#include <ATen/cuda/Exceptions.h>
#include <ATen/cuda/Atomic.cuh>
#include <c10/cuda/CUDAStream.h>
#include <c10/cuda/CUDACachingAllocator.h>

namespace {
template <typename T>
__host__ __device__ __forceinline__ T impartail_ceil_div(T a, T b) {
  return (a + b - 1) / b;
}
}  // namespace"""

SUBS = [
    # include cũ -> bộ header hiện đại (chỉ chèn một lần cho tệp đầu tiên khớp)
    (re.compile(r"#include <THC/THC\.h>\n(#include <THC/THCAtomics\.cuh>\n)?"
                r"(#include <THC/THCDeviceUtils\.cuh>\n)?"), HEADERS + "\n"),
    (re.compile(r"#include <THC/[A-Za-z]+\.cuh>\n"), ""),
    (re.compile(r"\bTHCCeilDiv\b"), "impartail_ceil_div"),
    (re.compile(r"\bTHCudaCheck\b"), "AT_CUDA_CHECK"),
    (re.compile(r"THCudaMalloc\(state,\s*"),
     "c10::cuda::CUDACachingAllocator::raw_alloc("),
    (re.compile(r"THCudaFree\(state,\s*"),
     "c10::cuda::CUDACachingAllocator::raw_delete("),
    # dòng khởi tạo state không còn ý nghĩa
    (re.compile(r"^\s*THCState\s*\*\s*state\s*=.*$", re.M), ""),

    # ── API Tensor cũ, bỏ trong torch 1.x → 2.x ──────────────────────
    # Kiểm trực tiếp trên nguồn tag v2.4.1: TensorBase.h không còn khai báo
    # `.type()`, và `.data<T>()` đã thay bằng `data_ptr<T>()`.
    #   x.type().is_cuda()  -> x.is_cuda()          (25 + 18 chỗ)
    #   x.data<T>()         -> x.data_ptr<T>()      (26 chỗ)
    #   a.type() == b.type()-> a.scalar_type() == b.scalar_type()
    #   AT_CHECK            -> TORCH_CHECK
    (re.compile(r"\.type\(\)\.is_cuda\(\)"), ".is_cuda()"),
    (re.compile(r"(\w+)\.type\(\)\s*==\s*(\w+)\.type\(\)"),
     r"\1.scalar_type() == \2.scalar_type()"),
    (re.compile(r"AT_DISPATCH_FLOATING_TYPES\(\s*(\w+)\.type\(\)"),
     r"AT_DISPATCH_FLOATING_TYPES(\1.scalar_type()"),
    (re.compile(r"\.data<([A-Za-z_][A-Za-z0-9_]*)>\(\)"), r".data_ptr<\1>()"),
    (re.compile(r"\bAT_CHECK\b"), "TORCH_CHECK"),
]


def patch(path):
    src = open(path, encoding="utf-8").read()
    out = src
    for pat, rep in SUBS:
        out = pat.sub(rep, out)
    if out == src:
        return 0
    open(path, "w", encoding="utf-8").write(out)
    left = len(re.findall(r"\bTHC[A-Za-z_]*\b", out))
    print(f"  ✓ {os.path.basename(path):22} còn {left} ký hiệu THC")
    return 1



ROOT = "/kaggle/working/ImparTail"
csrc = os.path.join(ROOT, "fasterRCNN/lib/model/csrc")
n = 0
for dp, _, fn in os.walk(csrc):
    for f in fn:
        if f.endswith((".cu", ".cuh", ".h", ".cpp")):
            n += patch(os.path.join(dp, f))
left = sum(len(re.findall(r"\bTHC[A-Za-z_]*\b", open(os.path.join(dp,f),encoding="utf-8").read()))
           for dp,_,fns in os.walk(csrc) for f in fns
           if f.endswith((".cu",".cuh",".h",".cpp")))
print(f"  đã vá {n} tệp | còn {left} tham chiếu THC")

env = dict(os.environ, MAX_JOBS="1", TORCH_CUDA_ARCH_LIST="6.0")
r = subprocess.run([sys.executable, "setup.py", "build", "develop"],
                   cwd=f"{ROOT}/fasterRCNN/lib", capture_output=True, text=True, env=env)
full = r.stdout + r.stderr
errs = [l for l in full.split("\n")
        if re.search(r"\berror\b|fatal error|no member|undefined|not declared|"
                     r"No such file|ninja: build stopped", l, re.I)]
print(f"\n=== biên dịch: exit {r.returncode} ===")
for l in errs[:25]: print("  " + l.strip()[:150])
if not errs: print("  không có dòng lỗi nào")

chk = subprocess.run([sys.executable, "-c",
       "import sys; sys.path.insert(0,'/kaggle/working/ImparTail');"
       "from fasterRCNN.lib.model.roi_layers import nms; print('nms OK')"],
      capture_output=True, text=True, cwd=ROOT)
print("\n✓✓ nms IMPORT ĐƯỢC — rào cản chính đã gỡ" if "nms OK" in chk.stdout
      else f"\n✗ nms hỏng: {chk.stderr.strip().split(chr(10))[-1][:140]}")

In [ ]:
# ── 6. Thư viện ────────────────────────────────────────────────────
# Danh sách này do QUÉT CÂY IMPORT ở máy mà ra (32 module cục bộ từ
# test_sgg_methods.py), không phải cài thử từng cái theo lỗi.
#
# pyrebase: chỉ để tác giả đẩy kết quả lên Firebase. Dòng gọi nó ĐÃ BỊ COMMENT
# (test_sgg_base.py:158 `# self._publish_results_to_firebase()`) nhưng import ở
# đầu tệp thì còn, nên vẫn chặn. Không cài được thì vô hiệu hoá import — nó
# không liên quan gì tới đánh giá.
import subprocess, sys, os, importlib.util as iu

PKGS = ["opencv-python-headless", "dill", "easydict", "h5py", "matplotlib",
        "scikit-image", "Wand", "wandb", "torchdiffeq", "torchsde"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + PKGS,
               capture_output=True, text=True)

# torchvision phải khớp torch 2.4.1
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "torchvision==0.19.1", "--index-url",
                "https://download.pytorch.org/whl/cu121"], capture_output=True, text=True)

# Firebase: GỠ VÔ ĐIỀU KIỆN, không thử cài.
# pyrebase4 cài được nhưng import của nó kéo theo pyOpenSSL đời cũ và chết ở
#   AttributeError: module 'lib' has no attribute 'GEN_EMAIL'
# Kiểm ở máy: FirebaseService() chỉ được gọi trong _publish_results_to_firebase(),
# và MỌI lời gọi hàm đó đều đã bị tác giả comment
# (test_sgg_base.py:158, test_sga_base.py:210). Nên gỡ import là an toàn tuyệt đối
# — nó chỉ để đẩy kết quả lên cloud, không dính gì tới đánh giá.
for f in ("test_sgg_base.py", "test_sga_base.py", "test_easg_base.py"):
    fp = f"/kaggle/working/ImparTail/{f}"
    if not os.path.exists(fp): continue
    s_ = open(fp, encoding="utf-8").read()
    s2 = s_.replace("from analysis.results.FirebaseService import FirebaseService",
                    "FirebaseService = None  # gỡ: chỉ dùng để đẩy cloud, đã tắt sẵn")
    if s2 != s_:
        open(fp, "w", encoding="utf-8").write(s2)
        print(f"  ✓ gỡ import Firebase khỏi {f}")

print("  === kiểm import ===")
for m in ("cv2","dill","easydict","h5py","matplotlib","skimage","wand",
          "wandb","torchvision","networkx","PIL","scipy","yaml","tqdm"):
    chk = subprocess.run([sys.executable, "-c", f"import {m}"],
                         capture_output=True, text=True)
    print(f"    {'✓' if chk.returncode==0 else '✗'} {m}"
          + ("" if chk.returncode==0 else f"  {chk.stderr.strip().split(chr(10))[-1][:70]}"))

# ── Bù module `analysis` thiếu trong kho mã ─────────────────────────
# Chạy thử ở máy phát hiện: ba tệp test đều import từ analysis.results, nhưng
# thư mục analysis/ KHÔNG CÓ trong kho mã công bố (và cũng không trong
# .gitignore). Thiếu nó thì script dừng ngay ở import.
# Ba lớp đó chỉ dùng trong _prepare_metrics_from_stats() và
# _publish_results_to_firebase() — nhánh Firebase đã bị comment sẵn — nên module
# giả không đổi con số đánh giá nào.
#!/usr/bin/env python3
"""Tạo module `analysis` còn thiếu trong kho mã ImparTail.

Ba tệp test (`test_sgg_base.py`, `test_sga_base.py`, `test_easg_base.py`) đều
import từ `analysis.results`, nhưng **thư mục `analysis/` không có trong kho mã
đã công bố** và cũng không nằm trong `.gitignore`. Thiếu nó thì mọi script đánh
giá dừng ngay ở dòng import, trước khi nạp được mô hình.

Kiểm ở máy cho thấy `Result`, `ResultDetails`, `Metrics` chỉ được dùng bên trong
`_prepare_metrics_from_stats()` và `_publish_results_to_firebase()` — cả hai đều
thuộc nhánh đẩy kết quả lên Firebase, mà mọi lời gọi nhánh đó đã bị tác giả
comment (`test_sgg_base.py:158`, `test_sga_base.py:210`).

Nên module giả chỉ cần thoả import. Nó **không** thay đổi con số nào của phép
đánh giá; nếu có mã nào thật sự dùng tới thì các lớp này vẫn ghi nhận thuộc tính
bình thường thay vì lỗi âm thầm.

    python stub_analysis.py <đường dẫn ImparTail>
"""
import os, sys

RESULT_PY = '''"""Bù cho module `analysis.results` không có trong kho mã công bố.

Chỉ đủ để thoả import. Xem impartail/patches/stub_analysis.py để biết vì sao.
"""


class _Bag:
    """Nhận mọi keyword, giữ lại, và cho gán thêm thuộc tính về sau."""

    def __init__(self, **kw):
        self.__dict__.update(kw)

    def __getattr__(self, name):
        # phương thức add_* / set_* gọi được mà không nổ
        if name.startswith(("add_", "set_", "publish_")):
            return lambda *a, **k: None
        raise AttributeError(name)

    def __repr__(self):
        return f"{type(self).__name__}({self.__dict__})"


class Metrics(_Bag):
    pass


class ResultDetails(_Bag):
    pass


class Result(_Bag):
    pass
'''

FIREBASE_PY = '''"""Bù cho `analysis.results.FirebaseService`.

Dịch vụ thật đẩy kết quả lên Firebase và cần `pyrebase`, thư viện kéo theo một
bản `pyOpenSSL` cũ không chạy được với `cryptography` hiện tại. Mọi lời gọi nó
trong kho mã đã bị comment sẵn, nên bản giả này chỉ để import không lỗi.
"""


class FirebaseService:
    def __init__(self, *a, **k):
        raise RuntimeError(
            "FirebaseService bị vô hiệu hoá: chỉ dùng để đẩy kết quả lên cloud, "
            "không liên quan tới đánh giá."
        )
'''


def main(root):
    d = os.path.join(root, "analysis", "results")
    if os.path.isdir(os.path.join(root, "analysis")) and not os.environ.get("FORCE"):
        print("  analysis/ đã tồn tại — không ghi đè")
        return 0
    os.makedirs(d, exist_ok=True)
    for pkg in (os.path.join(root, "analysis"), d):
        open(os.path.join(pkg, "__init__.py"), "w").close()
    open(os.path.join(d, "Result.py"), "w", encoding="utf-8").write(RESULT_PY)
    open(os.path.join(d, "FirebaseService.py"), "w", encoding="utf-8").write(FIREBASE_PY)
    print(f"  ✓ tạo analysis/results/ với Result.py và FirebaseService.py")
    return 0



main("/kaggle/working/ImparTail")


In [ ]:
# ── 7. Checkpoint: tải thẳng từ UT Dallas Box ───────────────────────
# Box KHÔNG chặn tải tự động — endpoint box_download_shared_file dùng được,
# đã kiểm ở máy: 3 tệp, 1,1 GB, đúng định dạng torch pickle.
#   sttran_predcls / sttran_sgcls : sharedName ji6aek…, file_id f_16742…
#   faster_rcnn_ag                : sharedName 1pspm5…, file_id f_16600…
import os, subprocess, glob
CK = "/kaggle/working/ckpt"; os.makedirs(CK, exist_ok=True)
BOX = "https://utdallas.box.com/index.php?rm=box_download_shared_file"
FILES = [
  ("faster_rcnn_ag.pth",  "1pspm5x8etlczoklyw4bclotsmlejagj", "f_1660078888773"),
  ("sttran_predcls.tar",  "ji6aekd4d8iguqk59rfwcalhpu608mop", "f_1674218919601"),
  ("sttran_sgcls.tar",    "ji6aekd4d8iguqk59rfwcalhpu608mop", "f_1674209087058"),
]
for name, shared, fid in FILES:
    dst = f"{CK}/{name}"
    if os.path.exists(dst) and os.path.getsize(dst) > 1e8:
        print(f"  đã có {name} ({os.path.getsize(dst)/1e6:.0f} MB)"); continue
    subprocess.run(["curl", "-sL", "-o", dst, "--max-time", "600",
                    f"{BOX}&shared_name={shared}&file_id={fid}"], check=False)
    sz = os.path.getsize(dst) if os.path.exists(dst) else 0
    print(f"  {'✓' if sz > 1e8 else '✗'} {name}: {sz/1e6:.0f} MB")

# fasterRCNN đọc checkpoint ở đường dẫn cố định trong lib/object_detector.py
import re as _re
od = "/kaggle/working/ImparTail/lib/object_detector.py"
src = open(od, encoding="utf-8").read()
src2 = _re.sub(r'["\'][^"\']*faster_rcnn_ag\.pth["\']', f'"{CK}/faster_rcnn_ag.pth"', src)
if src2 != src:
    open(od, "w", encoding="utf-8").write(src2)
    print("  ✓ trỏ object_detector.py sang checkpoint vừa tải")
print("  ckpt:", sorted(os.listdir(CK)))

In [ ]:
# ── 8. CHẨN ĐOÁN — chạy được tới đâu ────────────────────────────────
import os, glob, importlib, traceback
rows = []
def chk(name, fn):
    try: ok, note = fn()
    except Exception as e: ok, note = False, f"{type(e).__name__}: {str(e)[:60]}"
    rows.append((name, ok, note)); return ok

chk("khung hình đã gắn",
    lambda: (len(os.listdir(f"{AG}/frames")) > 100,
             f"{len(os.listdir(f'{AG}/frames'))} video"))
_ANN_NEEDED = ["person_bbox.pkl", "object_bbox_and_relationship.pkl",
               "object_classes.txt", "relationship_classes.txt"]
chk("annotation đã gắn",
    lambda: (all(os.path.exists(f"{AG}/annotations/{f}") for f in _ANN_NEEDED),
             ", ".join(f for f in _ANN_NEEDED
                       if not os.path.exists(f"{AG}/annotations/{f}")) or "đủ 4 tệp"))
chk("Cython (2 extension)",
    lambda: (bool(glob.glob("/kaggle/working/ImparTail/lib/draw_rectangles/*.so")) and
             bool(glob.glob("/kaggle/working/ImparTail/lib/fpn/box_intersections_cpu/*.so")),
             "draw_rectangles + bbox"))
def _nms():
    # PHẢI kiểm qua subprocess. Tiến trình notebook đã nạp torch 2.10 từ lúc
    # khởi động; ô 0b cài torch 2.4.1 nhưng bản cũ vẫn nằm trong bộ nhớ. Extension
    # biên dịch cho 2.4.1 sẽ không nạp được vào tiến trình đang giữ 2.10 —
    # import ở đây cho kết quả SAI (đã bị đúng lỗi này ở v6).
    import subprocess, sys
    r = subprocess.run([sys.executable, "-c",
         "import sys; sys.path.insert(0,'/kaggle/working/ImparTail');"
         "from fasterRCNN.lib.model.roi_layers import nms; print('OK')"],
        capture_output=True, text=True, cwd="/kaggle/working/ImparTail")
    if "OK" in r.stdout:
        return True, "import OK (subprocess)"
    return False, (r.stderr.strip().split("\n")[-1][:100] if r.stderr else "lỗi không rõ")
chk("fasterRCNN CUDA ext", _nms)
chk("checkpoint tác giả",
    lambda: (bool(glob.glob("/kaggle/working/ckpt/*.tar") + glob.glob("/kaggle/working/ckpt/*.pth")), "dataset impartail-ckpt-ag"))

print(f"  {'bước':<26} {'':3} ghi chú")
for n, ok, note in rows:
    print(f"  {n:<26} {'✓' if ok else '✗':3} {note}")

blockers = [n for n, ok, _ in rows if not ok]
print()
if blockers:
    print("CHẶN Ở:", ", ".join(blockers))
    print("\nKhông chạy được đánh giá. Đây là kết quả có giá trị — nó cho biết")
    print("chính xác rào cản nằm ở đâu, thay vì phỏng đoán.")
else:
    print("Mọi thứ sẵn sàng — chạy ô tiếp theo.")

In [ ]:
# ── 8b. Áp bản vá công thức (5) ─────────────────────────────────────
# Bản vá nhúng dưới dạng base64: repo là private nên raw.githubusercontent trả
# 404, còn nhúng thẳng mã nguồn thì dấu '''"""''' trong docstring của nó làm vỡ
# chuỗi bọc ngoài (đã dính đúng lỗi này khi dựng notebook).
#
# Bản vá thêm đường thứ hai sau cờ --true_masked_bce; KHÔNG bật cờ thì hành vi
# giống hệt mã gốc, nên hai nhánh so sánh được công bằng.
import base64, os, py_compile, subprocess, sys

PATCH_B64 = (
    "IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiJDw6BpIG3hurd0IG7huqEgxJHDum5nIHRoZW8gY8O0"
    "bmcgdGjhu6ljICg1KSBj4bunYSBJbXBhclRhaWwsIHNhdSBj4budIC0tdHJ1ZV9tYXNrZWRfYmNl"
    "LgoKQsOgaSBiw6FvLCBt4bulYyA0LjIuMSwgY8O0bmcgdGjhu6ljICg1KSwga8OobSBjw6J1ICJB"
    "IG1hc2sgLi4uIHZvaWRzIHRoZSBjb250cmlidXRpb24Kb2YgbG9zcyI6CgogICAgTCA9ICgxIC0g"
    "bSkgKiBMID0geyBMIG7hur91IG09MCA7IDAgbuG6v3UgbT0xIH0KCk5ow6NuIGLhu4sgY2hlIHBo"
    "4bqjaSDEkcOzbmcgZ8OzcCDEkMOaTkcgQuG6sE5HIDAuCgpNw6MgY8O0bmcgYuG7kSAoc3RzZ19i"
    "YXNlLnB5LCBuaMOhbmggQkNFKSBk4buxbmcgdmVjdG9yIMSRw61jaCB0b8OgbiAwIHLhu5NpIGNo"
    "4buJIHNldCAxIGNobwpjw6FjIG5ow6NuIFPhu5BORyBTw5NULiBWw6wgbm4uQkNFTG9zcygpIGTD"
    "uW5nIHJlZHVjdGlvbj0nbWVhbicgdHLDqm4gdG/DoG4gYuG7mSBtYXhfbGVuCnBo4bqnbiB04but"
    "LCBuaMOjbiBkxrDGoW5nIGLhu4sgY2hlIHRy4bufIHRow6BuaCBuaMOjbiDDgk0gdsOgIHbhuqtu"
    "IHNpbmggZ3JhZGllbnQuIEhhaSBzYWkgbOG7h2NoOgoKICAoMSkgZOG6pXUgICAgLSBuaMOjbiBi"
    "4buLIGNoZSBi4buLIMSR4bqpeSB4w6FjIHN14bqldCBYVeG7kE5HIHRoYXkgdsOsIMSRxrDhu6Nj"
    "IGLhu48gcXVhOwogICgyKSBt4bqrdSBz4buRIC0gY2hpYSBtYXhfbGVuIHRoYXkgdsOsIHPhu5Eg"
    "cGjhuqduIHThu60gY8OybiBoaeG7h3UgbOG7sWMsIGNvIG3hu41pIGdyYWRpZW50CiAgICAgICAg"
    "ICAgICAgIGPDsm4gbOG6oWkgKHbDrSBk4bulIDE1LzE3ID0gMC44OHgpLgoKQuG6o24gdsOhIHRo"
    "w6ptIG3hu5l0IMSRxrDhu51uZyB0aOG7qSBoYWksIGLhuq10IGLhurFuZyAtLXRydWVfbWFza2Vk"
    "X2JjZSwgZOG7sW5nIG3hurd0IG7huqEg4bufIGPhuqVwClBI4bqmTiBU4busIHLhu5NpIHTDrW5o"
    "IEJDRSBjw7MgbeG6t3QgbuG6oS4gTeG6t2MgxJHhu4tuaCBU4bquVDoga2jDtG5nIHRydXnhu4Fu"
    "IGPhu50gdGjDrCBow6BuaCB2aQpnaeG7kW5nIGjhu4d0IG3DoyBn4buRYywgxJHhu4MgaGFpIG5o"
    "w6FuaCBzbyBzw6FuaCDEkcaw4bujYyBjw7RuZyBi4bqxbmcuCgpHaGkgY2jDuiB24buBIHBow6F0"
    "IGJp4buDdTogxJHDonkgbMOgIMSR4buRaSBjaGnhur91IE3DgyDEkMODIEPDlE5HIELhu5AgduG7"
    "m2kgQ8OUTkcgVEjhu6hDIMSQw4MgQ8OUTkcgQuG7kCwKa2jDtG5nIHBo4bqjaSBr4bq/dCBsdeG6"
    "rW4gduG7gSB2aeG7h2MgdMOhYyBnaeG6oyDEkcOjIGNo4bqheSB0aMOtIG5naGnhu4dtIHRo4bq/"
    "IG7DoG8uCgpEw7luZzogcHl0aG9uIGZpeF9tYXNrZWRfYmNlLnB5IC/Ekcaw4budbmcvZOG6q24v"
    "SW1wYXJUYWlsCiIiIgppbXBvcnQgb3MKaW1wb3J0IHJlCmltcG9ydCBzeXMKCiMg4pSA4pSAIDEu"
    "IHN0c2dfYmFzZS5weTogZOG7sW5nIHRow6ptIG3hurd0IG7huqEgY+G6pXAgcGjhuqduIHThu60g"
    "4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACkNV"
    "X1NUU0cgPSAnJycgICAgICAgICAgICAgICAgaWYgZ3RfbWFza2VkLnNoYXBlWzBdID09IDA6CiAg"
    "ICAgICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgICAgIGVsc2U6CiAgICAgICAg"
    "ICAgICAgICAgICAgbGFiZWwgPSB0b3JjaC56ZXJvcyhbbWF4X2xlbl0sIGR0eXBlPXRvcmNoLmZs"
    "b2F0MzIsIGRldmljZT1zZWxmLl9kZXZpY2UpCiAgICAgICAgICAgICAgICAgICAgbGFiZWxbZ3Rf"
    "bWFza2VkXSA9IDEKICAgICAgICAgICAgICAgICAgICBmaWx0ZXJlZF9sYWJlbHMuYXBwZW5kKGxh"
    "YmVsKQogICAgICAgICAgICAgICAgICAgIGZpbHRlcmVkX2Rpc3RyaWJ1dGlvbi5hcHBlbmQocHJl"
    "ZF9kaXN0cmlidXRpb25faSknJycKCk1PSV9TVFNHID0gJycnICAgICAgICAgICAgICAgIGlmIGd0"
    "X21hc2tlZC5zaGFwZVswXSA9PSAwOgogICAgICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAg"
    "ICAgICAgICAgICBlbHNlOgogICAgICAgICAgICAgICAgICAgIGxhYmVsID0gdG9yY2guemVyb3Mo"
    "W21heF9sZW5dLCBkdHlwZT10b3JjaC5mbG9hdDMyLCBkZXZpY2U9c2VsZi5fZGV2aWNlKQogICAg"
    "ICAgICAgICAgICAgICAgIGxhYmVsW2d0X21hc2tlZF0gPSAxCiAgICAgICAgICAgICAgICAgICAg"
    "ZmlsdGVyZWRfbGFiZWxzLmFwcGVuZChsYWJlbCkKICAgICAgICAgICAgICAgICAgICBmaWx0ZXJl"
    "ZF9kaXN0cmlidXRpb24uYXBwZW5kKHByZWRfZGlzdHJpYnV0aW9uX2kpCgogICAgICAgICAgICAg"
    "ICAgICAgICMgQ8O0bmcgdGjhu6ljICg1KTogbmjDo24gYuG7iyBjaGUgcGjhuqNpIMSRw7NuZyBn"
    "w7NwIDAuIOG7niBuaMOhbmggZ+G7kWMKICAgICAgICAgICAgICAgICAgICAjIGNow7puZyB0aMOg"
    "bmggMCA9IG5ow6NuIMOibSB2w6AgduG6q24gc2luaCBncmFkaWVudC4gTeG6t3QgbuG6oSBj4bql"
    "cAogICAgICAgICAgICAgICAgICAgICMgcGjhuqduIHThu60gZMaw4bubaSDEkcOieSBjaG8gcGjD"
    "qXAgbG/huqFpIMSRw7puZyBjaMO6bmcga2jhu49pIHBow6lwIHTDrW5oLgogICAgICAgICAgICAg"
    "ICAgICAgIGVsID0gdG9yY2gub25lcyhbbWF4X2xlbl0sIGR0eXBlPXRvcmNoLmZsb2F0MzIsCiAg"
    "ICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGRldmljZT1zZWxmLl9kZXZpY2UpCiAg"
    "ICAgICAgICAgICAgICAgICAgaWYgZ3QubnVtZWwoKSA+IDA6CiAgICAgICAgICAgICAgICAgICAg"
    "ICAgIGVsW2d0W21hc2sgPT0gMF1dID0gMC4wCiAgICAgICAgICAgICAgICAgICAgZWxlbWVudF9t"
    "YXNrcy5hcHBlbmQoZWwpJycnCgojIOKUgOKUgCAyLiBraOG7n2kgdOG6oW8gdsOgIHRy4bqjIHbh"
    "u4EgZWxlbWVudF9tYXNrcyDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi"
    "lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKQ1VfSU5JVCA9ICcn"
    "JyAgICAgICAgZmlsdGVyZWRfbGFiZWxzID0gW10KICAgICAgICBmaWx0ZXJlZF9kaXN0cmlidXRp"
    "b24gPSBbXQogICAgICAgIGlmIG5vdCBzZWxmLl9jb25mLmJjZV9sb3NzOicnJwpNT0lfSU5JVCA9"
    "ICcnJyAgICAgICAgZmlsdGVyZWRfbGFiZWxzID0gW10KICAgICAgICBmaWx0ZXJlZF9kaXN0cmli"
    "dXRpb24gPSBbXQogICAgICAgIGVsZW1lbnRfbWFza3MgPSBbXQogICAgICAgIGlmIG5vdCBzZWxm"
    "Ll9jb25mLmJjZV9sb3NzOicnJwoKQ1VfUkVUX05PTkUgPSAnJycgICAgICAgIGlmIGxlbihmaWx0"
    "ZXJlZF9sYWJlbHMpID09IDAgYW5kIGxlbihmaWx0ZXJlZF9kaXN0cmlidXRpb24pID09IDA6CiAg"
    "ICAgICAgICAgIHJldHVybiBOb25lLCBOb25lLCBwcmVkX2Rpc3RyaWJ1dGlvbicnJwpNT0lfUkVU"
    "X05PTkUgPSAnJycgICAgICAgIGlmIGxlbihmaWx0ZXJlZF9sYWJlbHMpID09IDAgYW5kIGxlbihm"
    "aWx0ZXJlZF9kaXN0cmlidXRpb24pID09IDA6CiAgICAgICAgICAgIHNlbGYuX2xhc3RfZWxlbWVu"
    "dF9tYXNrID0gTm9uZQogICAgICAgICAgICByZXR1cm4gTm9uZSwgTm9uZSwgcHJlZF9kaXN0cmli"
    "dXRpb24nJycKCkNVX1JFVCA9ICcnJyAgICAgICAgZmlsdGVyZWRfbGFiZWxzID0gdG9yY2guc3Rh"
    "Y2soZmlsdGVyZWRfbGFiZWxzKQogICAgICAgIGZpbHRlcmVkX2Rpc3RyaWJ1dGlvbiA9IHRvcmNo"
    "LnN0YWNrKGZpbHRlcmVkX2Rpc3RyaWJ1dGlvbikKCiAgICAgICAgcmV0dXJuIGZpbHRlcmVkX2Rp"
    "c3RyaWJ1dGlvbiwgZmlsdGVyZWRfbGFiZWxzLCBwcmVkX2Rpc3RyaWJ1dGlvbicnJwpNT0lfUkVU"
    "ID0gJycnICAgICAgICBmaWx0ZXJlZF9sYWJlbHMgPSB0b3JjaC5zdGFjayhmaWx0ZXJlZF9sYWJl"
    "bHMpCiAgICAgICAgZmlsdGVyZWRfZGlzdHJpYnV0aW9uID0gdG9yY2guc3RhY2soZmlsdGVyZWRf"
    "ZGlzdHJpYnV0aW9uKQogICAgICAgICMgVHJ1eeG7gW4gcXVhIHRodeG7mWMgdMOtbmggdGhheSB2"
    "w6wgxJHhu5VpIGNo4buvIGvDvSBow6BtOiBow6BtIG7DoHkgxJHGsOG7o2MgZ+G7jWkgdOG7qwog"
    "ICAgICAgICMgbmhp4buBdSBuxqFpIChTR0cgdsOgIFNHQSksIMSR4buVaSBjaOG7ryBrw70gc+G6"
    "vSBwaOG6o2kgc+G7rWEgaOG6v3QuCiAgICAgICAgc2VsZi5fbGFzdF9lbGVtZW50X21hc2sgPSB0"
    "b3JjaC5zdGFjayhlbGVtZW50X21hc2tzKSBpZiBlbGVtZW50X21hc2tzIGVsc2UgTm9uZQoKICAg"
    "ICAgICByZXR1cm4gZmlsdGVyZWRfZGlzdHJpYnV0aW9uLCBmaWx0ZXJlZF9sYWJlbHMsIHByZWRf"
    "ZGlzdHJpYnV0aW9uJycnCgojIOKUgOKUgCAzLiB0cmFpbl9zZ2dfYmFzZS5weTogZMO5bmcgbeG6"
    "t3QgbuG6oSBraGkgY+G7nSDEkcaw4bujYyBi4bqtdCDilIDilIDilIDilIDilIDilIDilIDilIDi"
    "lIDilIDilIDilIDilIDilIAKQ1VfTE9TUyA9ICcnJyAgICAgICAgICAgIGVsc2U6CiAgICAgICAg"
    "ICAgICAgICBsb3NzZXNbY29uc3Que1RFTn1fUkVMQVRJT05fTE9TU10gPSBzZWxmLl9iY2VfbG9z"
    "cyhmaWx0ZXJlZF97dn1fZGlzdHJpYnV0aW9uLCBmaWx0ZXJlZF97dn1fbGFiZWxzKScnJwpNT0lf"
    "TE9TUyA9ICcnJyAgICAgICAgICAgIGVsaWYgZ2V0YXR0cihzZWxmLl9jb25mLCAidHJ1ZV9tYXNr"
    "ZWRfYmNlIiwgRmFsc2UpOgogICAgICAgICAgICAgICAgbG9zc2VzW2NvbnN0LntURU59X1JFTEFU"
    "SU9OX0xPU1NdID0gc2VsZi5fbWFza2VkX2JjZSgKICAgICAgICAgICAgICAgICAgICBmaWx0ZXJl"
    "ZF97dn1fZGlzdHJpYnV0aW9uLCBmaWx0ZXJlZF97dn1fbGFiZWxzLCB7dn1fZWxlbWVudF9tYXNr"
    "KQogICAgICAgICAgICBlbHNlOgogICAgICAgICAgICAgICAgbG9zc2VzW2NvbnN0LntURU59X1JF"
    "TEFUSU9OX0xPU1NdID0gc2VsZi5fYmNlX2xvc3MoZmlsdGVyZWRfe3Z9X2Rpc3RyaWJ1dGlvbiwg"
    "ZmlsdGVyZWRfe3Z9X2xhYmVscyknJycKCiMgaMOgbSBfbWFza2VkX2JjZSwgxJHhurd0IGPhuqFu"
    "aCBfaW5pdF9sb3NzX2Z1bmN0aW9ucwpDVV9MT1NTRk4gPSAnJycgICAgZGVmIF9pbml0X2xvc3Nf"
    "ZnVuY3Rpb25zKHNlbGYpOgogICAgICAgIHNlbGYuX2JjZV9sb3NzID0gbm4uQkNFTG9zcygpJycn"
    "Ck1PSV9MT1NTRk4gPSAnJycgICAgZGVmIF9tYXNrZWRfYmNlKHNlbGYsIHByZWRfZGlzdHJpYnV0"
    "aW9uLCBsYWJlbHMsIGVsZW1lbnRfbWFzayk6CiAgICAgICAgIiIiQkNFIHTDtG4gdHLhu41uZyBj"
    "w7RuZyB0aOG7qWMgKDUpOiBwaOG6p24gdOG7rSBi4buLIGNoZSDEkcOzbmcgZ8OzcCDEkcO6bmcg"
    "MC4KCiAgICAgICAgS2jDoWMgaGFpIMSRaeG7g20gc28gduG7m2kgbm4uQkNFTG9zcygpIG3hurdj"
    "IMSR4buLbmg6CiAgICAgICAgICAtIHJlZHVjdGlvbj0nbm9uZScgcuG7k2kgbmjDom4gbeG6t3Qg"
    "buG6oSAtPiBwaOG6p24gdOG7rSBi4buLIGNoZSBraMO0bmcgc2luaCBncmFkaWVudDsKICAgICAg"
    "ICAgIC0gY2h14bqpbiBob8OhIHRoZW8gc+G7kSBwaOG6p24gdOG7rSBDw5JOIEhJ4buGVSBM4buw"
    "Qywga2jDtG5nIHRoZW8gbWF4X2xlbi4KICAgICAgICAiIiIKICAgICAgICBpZiBlbGVtZW50X21h"
    "c2sgaXMgTm9uZToKICAgICAgICAgICAgcmV0dXJuIHNlbGYuX2JjZV9sb3NzKHByZWRfZGlzdHJp"
    "YnV0aW9uLCBsYWJlbHMpCiAgICAgICAgcGVyID0gbm4uZnVuY3Rpb25hbC5iaW5hcnlfY3Jvc3Nf"
    "ZW50cm9weSgKICAgICAgICAgICAgcHJlZF9kaXN0cmlidXRpb24sIGxhYmVscywgcmVkdWN0aW9u"
    "PSJub25lIikKICAgICAgICB0b25nID0gZWxlbWVudF9tYXNrLnN1bSgpCiAgICAgICAgaWYgdG9u"
    "Zy5pdGVtKCkgPT0gMDoKICAgICAgICAgICAgcmV0dXJuIHByZWRfZGlzdHJpYnV0aW9uLnN1bSgp"
    "ICogMC4wCiAgICAgICAgcmV0dXJuIChwZXIgKiBlbGVtZW50X21hc2spLnN1bSgpIC8gdG9uZwoK"
    "ICAgIGRlZiBfaW5pdF9sb3NzX2Z1bmN0aW9ucyhzZWxmKToKICAgICAgICBzZWxmLl9iY2VfbG9z"
    "cyA9IG5uLkJDRUxvc3MoKScnJwoKCmRlZiBkb2MocCk6CiAgICB3aXRoIG9wZW4ocCwgZW5jb2Rp"
    "bmc9InV0Zi04IikgYXMgZjoKICAgICAgICByZXR1cm4gZi5yZWFkKCkKCgpkZWYgZ2hpKHAsIHMp"
    "OgogICAgd2l0aCBvcGVuKHAsICJ3IiwgZW5jb2Rpbmc9InV0Zi04IikgYXMgZjoKICAgICAgICBm"
    "LndyaXRlKHMpCgoKZGVmIHZhX3N0c2cocm9vdCk6CiAgICBwID0gb3MucGF0aC5qb2luKHJvb3Qs"
    "ICJzdHNnX2Jhc2UucHkiKQogICAgcyA9IGRvYyhwKQogICAgIyBE4bqldSBoaeG7h3Ugbmjhuq1u"
    "IGJp4bq/dCDEkcOjIHbDoTogcGjhuqNpIGzDoCBjaHXhu5dpIENI4buIIGPDsyB0cm9uZyBi4bqj"
    "biB2w6EuIETDuW5nIGTDsm5nCiAgICAjIMSR4bqndSBj4bunYSBgbW9pYCBsw6Agc2FpIOKAlCB2"
    "4bubaSBraOG7kWkgbeG6t3QgbuG6oSwgZMOybmcgxJHhuqd1IHRyw7luZyBtw6MgZ+G7kWMgbsOq"
    "biBs4bqnbgogICAgIyBjaOG6oXkgdGjhu6kgaGFpIGNow6huIHRow6ptIG3hu5l0IGto4buRaSBu"
    "4buvYS4KICAgIGZvciBjdSwgbW9pLCB0ZW4sIGRhdSBpbiBbCiAgICAgICAgICAgIChDVV9JTklU"
    "LCBNT0lfSU5JVCwgImto4bufaSB04bqhbyBlbGVtZW50X21hc2tzIiwgImVsZW1lbnRfbWFza3Mg"
    "PSBbXSIpLAogICAgICAgICAgICAoQ1VfU1RTRywgTU9JX1NUU0csICJk4buxbmcgbeG6t3QgbuG6"
    "oSBj4bqlcCBwaOG6p24gdOG7rSIsICJlbGVtZW50X21hc2tzLmFwcGVuZChlbCkiKSwKICAgICAg"
    "ICAgICAgKENVX1JFVF9OT05FLCBNT0lfUkVUX05PTkUsICJ0cuG6oyBOb25lIiwgInNlbGYuX2xh"
    "c3RfZWxlbWVudF9tYXNrID0gTm9uZSIpLAogICAgICAgICAgICAoQ1VfUkVULCBNT0lfUkVULCAi"
    "dHLhuqMgZWxlbWVudF9tYXNrcyIsICJ0b3JjaC5zdGFjayhlbGVtZW50X21hc2tzKSIpXToKICAg"
    "ICAgICBpZiBkYXUgaW4gczoKICAgICAgICAgICAgcHJpbnQoZiIgIC0ge3Rlbn06IMSRw6MgdsOh"
    "IHRyxrDhu5tjIMSRw7MiKQogICAgICAgICAgICBjb250aW51ZQogICAgICAgIGlmIGN1IG5vdCBp"
    "biBzOgogICAgICAgICAgICBzeXMuZXhpdChmIiAg4pyXIGtow7RuZyBraOG7m3A6IHt0ZW59IikK"
    "ICAgICAgICBzID0gcy5yZXBsYWNlKGN1LCBtb2ksIDEpCiAgICAgICAgcHJpbnQoZiIgIOKckyB7"
    "dGVufSIpCiAgICBnaGkocCwgcykKCgpkZWYgdmFfdHJhaW4ocm9vdCk6CiAgICBwID0gb3MucGF0"
    "aC5qb2luKHJvb3QsICJ0cmFpbl9zZ2dfYmFzZS5weSIpCiAgICBzID0gZG9jKHApCgogICAgaWYg"
    "Il9tYXNrZWRfYmNlIiBub3QgaW4gczoKICAgICAgICBpZiBDVV9MT1NTRk4gbm90IGluIHM6CiAg"
    "ICAgICAgICAgIHN5cy5leGl0KCIgIOKclyBraMO0bmcga2jhu5twOiBfaW5pdF9sb3NzX2Z1bmN0"
    "aW9ucyIpCiAgICAgICAgcyA9IHMucmVwbGFjZShDVV9MT1NTRk4sIE1PSV9MT1NTRk4sIDEpCiAg"
    "ICAgICAgcHJpbnQoIiAg4pyTIHRow6ptIGjDoG0gX21hc2tlZF9iY2UiKQogICAgZWxzZToKICAg"
    "ICAgICBwcmludCgiICAtIF9tYXNrZWRfYmNlOiDEkcOjIGPDsyIpCgogICAgZm9yIHRlbiwgdiBp"
    "biBbKCJTUEFUSUFMIiwgInNwYXRpYWwiKSwgKCJDT05UQUNUSU5HIiwgImNvbnRhY3QiKV06CiAg"
    "ICAgICAgY3UgPSBDVV9MT1NTLmZvcm1hdChURU49dGVuLCB2PXYpCiAgICAgICAgbW9pID0gTU9J"
    "X0xPU1MuZm9ybWF0KFRFTj10ZW4sIHY9dikKICAgICAgICBpZiBmInt2fV9lbGVtZW50X21hc2si"
    "IGluIHM6CiAgICAgICAgICAgIHByaW50KGYiICAtIG5ow6FuaCB7dn06IMSRw6MgdsOhIikKICAg"
    "ICAgICAgICAgY29udGludWUKICAgICAgICBpZiBjdSBub3QgaW4gczoKICAgICAgICAgICAgc3lz"
    "LmV4aXQoZiIgIOKclyBraMO0bmcga2jhu5twIG5ow6FuaCB7dn0iKQogICAgICAgIHMgPSBzLnJl"
    "cGxhY2UoY3UsIG1vaSwgMSkKICAgICAgICBwcmludChmIiAg4pyTIG5ow6FuaCB7dn0iKQoKICAg"
    "ICMgbOG6pXkgbeG6t3QgbuG6oSBuZ2F5IHNhdSBt4buXaSBs4budaSBn4buNaSBfcHJlcGFyZV9s"
    "YWJlbHNfYW5kX2Rpc3RyaWJ1dGlvbgogICAgZm9yIHYsIGhhbmcgaW4gWygic3BhdGlhbCIsICJt"
    "YXhfbGVuPTYiKSwgKCJjb250YWN0IiwgIm1heF9sZW49MTciKV06CiAgICAgICAgbmVvID0gZiIg"
    "ICAgICAgICAgICB7aGFuZ31cbiAgICAgICAgKVxuIgogICAgICAgIHRoZW0gPSBmIiAgICAgICAg"
    "ICAgIHtoYW5nfVxuICAgICAgICApXG4gICAgICAgIHt2fV9lbGVtZW50X21hc2sgPSBnZXRhdHRy"
    "KHNlbGYsIFwiX2xhc3RfZWxlbWVudF9tYXNrXCIsIE5vbmUpXG4iCiAgICAgICAgaWYgZiJ7dn1f"
    "ZWxlbWVudF9tYXNrID0gZ2V0YXR0ciIgaW4gczoKICAgICAgICAgICAgY29udGludWUKICAgICAg"
    "ICBpZiBuZW8gbm90IGluIHM6CiAgICAgICAgICAgIHN5cy5leGl0KGYiICDinJcga2jDtG5nIHTD"
    "rG0gdGjhuqV5IG5lbyB7aGFuZ30iKQogICAgICAgIHMgPSBzLnJlcGxhY2UobmVvLCB0aGVtLCAx"
    "KQogICAgICAgIHByaW50KGYiICDinJMgbOG6pXkgbeG6t3QgbuG6oSBjaG8ge3Z9IikKCiAgICBn"
    "aGkocCwgcykKCgpkZWYgdmFfY29uZmlnKHJvb3QpOgogICAgcCA9IG9zLnBhdGguam9pbihyb290"
    "LCAibGliL3N1cGVydmlzZWQvY29uZmlnLnB5IikKICAgIHMgPSBkb2MocCkKICAgIGlmICJ0cnVl"
    "X21hc2tlZF9iY2UiIGluIHM6CiAgICAgICAgcHJpbnQoIiAgLSBj4budIC0tdHJ1ZV9tYXNrZWRf"
    "YmNlOiDEkcOjIGPDsyIpCiAgICAgICAgcmV0dXJuCiAgICBuZW8gPSAiICAgICAgICBwYXJzZXIu"
    "YWRkX2FyZ3VtZW50KCctLWJjZV9sb3NzJywgYWN0aW9uPSdzdG9yZV90cnVlJykiCiAgICBpZiBu"
    "ZW8gbm90IGluIHM6CiAgICAgICAgc3lzLmV4aXQoIiAg4pyXIGtow7RuZyB0w6xtIHRo4bqleSBu"
    "ZW8gLS1iY2VfbG9zcyIpCiAgICB0aGVtID0gKG5lbyArICJcbiAgICAgICAgIyBN4bq3YyDEkeG7"
    "i25oIFThuq5UOiBraMO0bmcgdHJ1eeG7gW4gY+G7nSB0aMOsIGjDoG5oIHZpIGdp4buRbmcgaOG7"
    "h3QgbcOjIGfhu5FjLlxuIgogICAgICAgICAgICAiICAgICAgICBwYXJzZXIuYWRkX2FyZ3VtZW50"
    "KCctLXRydWVfbWFza2VkX2JjZScsIGFjdGlvbj0nc3RvcmVfdHJ1ZScpIikKICAgIGdoaShwLCBz"
    "LnJlcGxhY2UobmVvLCB0aGVtLCAxKSkKICAgIHByaW50KCIgIOKckyB0aMOqbSBj4budIC0tdHJ1"
    "ZV9tYXNrZWRfYmNlIikKCgpkZWYgbWFpbigpOgogICAgcm9vdCA9IHN5cy5hcmd2WzFdIGlmIGxl"
    "bihzeXMuYXJndikgPiAxIGVsc2UgIi4iCiAgICBpZiBub3Qgb3MucGF0aC5pc2ZpbGUob3MucGF0"
    "aC5qb2luKHJvb3QsICJzdHNnX2Jhc2UucHkiKSk6CiAgICAgICAgc3lzLmV4aXQoZiJraMO0bmcg"
    "dGjhuqV5IHN0c2dfYmFzZS5weSB0cm9uZyB7cm9vdH0iKQogICAgcHJpbnQoInN0c2dfYmFzZS5w"
    "eToiKQogICAgdmFfc3RzZyhyb290KQogICAgcHJpbnQoInRyYWluX3NnZ19iYXNlLnB5OiIpCiAg"
    "ICB2YV90cmFpbihyb290KQogICAgcHJpbnQoImxpYi9zdXBlcnZpc2VkL2NvbmZpZy5weToiKQog"
    "ICAgdmFfY29uZmlnKHJvb3QpCiAgICBwcmludCgiXG5Yb25nLiBC4bqtdCBi4bqxbmcgLS10cnVl"
    "X21hc2tlZF9iY2U7IGtow7RuZyB0cnV54buBbiB0aMOsIGNo4bqheSBuaMawIG3DoyBn4buRYy4i"
    "KQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBtYWluKCkK"
)

PATCH = "/kaggle/working/fix_masked_bce.py"
with open(PATCH, "wb") as f:
    f.write(base64.b64decode(PATCH_B64))

r = subprocess.run([sys.executable, PATCH, "/kaggle/working/ImparTail"],
                   capture_output=True, text=True)
print(r.stdout or r.stderr)

# Kiểm bản vá đã vào đúng chỗ — không tin vào thông báo của chính nó
ok = True
src = open("/kaggle/working/ImparTail/stsg_base.py").read()
n = src.count("element_masks.append(el)")
print(f"  khối mặt nạ cấp phần tử : {n} (phải = 1)"); ok &= n == 1
src2 = open("/kaggle/working/ImparTail/train_sgg_base.py").read()
print(f"  hàm _masked_bce         : {'có' if '_masked_bce' in src2 else 'THIẾU'}")
ok &= "_masked_bce" in src2
m = src2.count("_element_mask = getattr")
print(f"  nhánh spatial + contact : {m} (phải = 2)"); ok &= m == 2
cfg = open("/kaggle/working/ImparTail/lib/supervised/config.py").read()
print(f"  cờ --true_masked_bce    : {'có' if 'true_masked_bce' in cfg else 'THIẾU'}")
ok &= "true_masked_bce" in cfg

for f in ["stsg_base.py", "train_sgg_base.py", "lib/supervised/config.py"]:
    try:
        py_compile.compile(f"/kaggle/working/ImparTail/{f}", doraise=True)
    except Exception as e:
        print(f"  ✗ {f}: {e}"); ok = False

PATCH_OK = ok
print("\n" + ("✓ bản vá sẵn sàng" if ok else "✗ BẢN VÁ HỎNG — không chạy tiếp"))

In [ ]:
# ── 9. Đối soát khung hình TRAIN với ảnh thực có ───────────────────
# v7 của notebook đánh giá đã chạy 20 phút rồi chết vì thiếu ảnh: cv2.imread
# trả None và prep_im_for_blob gọi .astype lên None (ag_dataset.py:22 -> blob.py:39).
# Huấn luyện tốn hàng giờ nên càng phải đối soát TRƯỚC.
import pickle, os, collections, random

with open(f"{AG}/annotations/object_bbox_and_relationship.pkl", "rb") as f:
    obj = pickle.load(f)

train_frames = [k for k, v in obj.items() if v[0]["metadata"]["set"] == "train"]
vids = collections.Counter(k.split("/")[0] for k in train_frames)
print(f"khung hình train : {len(train_frames):,}")
print(f"video train      : {len(vids):,}")

have = set(os.listdir(f"{AG}/frames"))
missing_vid = [v for v in vids if v not in have]
print(f"video THIẾU      : {len(missing_vid):,}")

random.seed(0)
sample = random.sample(train_frames, min(400, len(train_frames)))
miss = [n for n in sample if not os.path.exists(f"{AG}/frames/{n}")]
print(f"mẫu 400 khung hình, thiếu ảnh : {len(miss)} ({100*len(miss)/len(sample):.1f}%)")

DATA_OK = not (missing_vid or miss)
print("\n" + ("✓ đủ dữ liệu train" if DATA_OK
               else "⚠ THIẾU DỮ LIỆU — huấn luyện sẽ chết giữa chừng"))

# ước lượng thời gian từ tốc độ đo được ở lần đánh giá: 2,2 giây/video
gio = len(vids) * 2.2 / 3600
print(f"\nước tính: {gio:.1f} giờ/epoch (2,2 s/video đo được ở lần đánh giá)")
print(f"giới hạn phiên Kaggle 12 giờ -> chạy {int(11 // gio) or 1} epoch mỗi phiên")

In [ ]:
# ── 10. Huấn luyện ─────────────────────────────────────────────────
# NHANH là biến duy nhất cần đổi giữa hai kernel:
#   NHANH = False -> nhánh gốc   (không truyền cờ)
#   NHANH = True  -> nhánh đã vá (--true_masked_bce)
# Mọi thứ còn lại giữ y hệt để khác biệt duy nhất là cách áp mặt nạ.
NHANH_VA = False          # <<< ĐỔI Ở ĐÂY
SO_EPOCH = 2              # quota 30 giờ/tuần không đủ 4 epoch x 2 nhánh

import glob, os, subprocess, sys, time

TEN = "va" if NHANH_VA else "goc"
SAVE = f"/kaggle/working/save_{TEN}"
os.makedirs(SAVE, exist_ok=True)

if not (PATCH_OK and DATA_OK):
    print("bỏ qua — ô 8b hoặc ô 9 chưa sạch")
else:
    # Nối phiên: nếu đã có checkpoint epoch trước thì học tiếp từ đó.
    # stsg_base.py:51-52 tách tên theo "_".join(stem.split('_')[:-2]) nên tên
    # dạng sttran_partial_10_predcls_epoch_N.tar khôi phục đúng name và mode.
    cu = sorted(glob.glob(f"{SAVE}/*_epoch_*.tar"))
    cmd = [sys.executable, "train_sgg_methods.py",
           "--task_name", "sgg",
           "--method_name", "sttran",
           "--mode", "predcls",
           "--datasize", "large",
           "--data_path", "/kaggle/working/data/ag",
           "--save_path", SAVE,
           "--nepoch", str(SO_EPOCH),
           "--use_partial_annotations",
           "--partial_percentage", "10",
           "--bce_loss"]            # nhánh BCE — chỗ bản vá tác động
    if NHANH_VA:
        cmd.append("--true_masked_bce")
    if cu:
        cmd += ["--ckpt", cu[-1]]
        print(f"học tiếp từ {os.path.basename(cu[-1])}")

    print(" ".join(cmd), flush=True)
    t0 = time.time()
    r = subprocess.run(cmd, cwd="/kaggle/working/ImparTail",
                       capture_output=True, text=True)
    out = (r.stdout + r.stderr).split("\n")
    # bỏ dòng tiến độ, giữ phần có nghĩa
    loc = [l for l in out if "it/s" not in l and "s/it" not in l]
    print("\n".join(loc[-40:]))
    print(f"\nexit {r.returncode} — {(time.time()-t0)/3600:.2f} giờ")
    print("checkpoint:", sorted(os.listdir(SAVE)))

In [ ]:
# ── 11. Đánh giá checkpoint vừa huấn luyện ─────────────────────────
# Cùng lệnh đã dùng ở notebook đánh giá (đã cho 9 số khớp Bảng 1, lệch 0,74).
import glob, os, subprocess, sys

ck = sorted(glob.glob(f"{SAVE}/*_epoch_*.tar"))
if not ck:
    print("chưa có checkpoint")
else:
    cuoi = ck[-1]
    print("đánh giá:", os.path.basename(cuoi))
    cmd = [sys.executable, "test_sgg_methods.py",
           "--task_name", "sgg", "--method_name", "sttran",
           "--mode", "predcls", "--datasize", "large",
           "--data_path", "/kaggle/working/data/ag",
           "--save_path", SAVE,
           "--ckpt", cuoi]
    r = subprocess.run(cmd, cwd="/kaggle/working/ImparTail",
                       capture_output=True, text=True)
    out = [l for l in (r.stdout + r.stderr).split("\n")
           if "it/s" not in l and "s/it" not in l]
    print("\n".join(out[-25:]))
    print(f"exit {r.returncode}")

    for f in glob.glob("/kaggle/working/ImparTail/results/**/*.csv", recursive=True):
        print("\n" + f)
        print(open(f).read()[:1200])